# Predicción del rendimiento académico con Regresión Lineal

En este notebook construirás un modelo **introductorio** de predicción de notas estudiantiles usando **scikit-learn**.

El dataset proviene de Kaggle y describe características de estudiantes (horas de estudio, asistencia, sueño, etc.) junto con su **Performance Index** (índice de rendimiento académico), que es una variable continua. Esto lo convierte en un problema clásico de **regresión**.

Aprenderás a:
1. Cargar y explorar datos reales
2. Separar correctamente en entrenamiento y prueba
3. Construir un **Pipeline** con **ColumnTransformer** para preprocesar variables numéricas y categóricas
4. Entrenar una **Regresión Lineal**
5. Evaluar el modelo con métricas apropiadas para regresión (MAE, RMSE, R²)
6. Interpretar los coeficientes del modelo
7. Experimentar con cambios (ejercicios al final)

> **¿Por qué Regresión Lineal?**  
> Cuando la variable objetivo es **continua** (un número real, como una nota de 0 a 100), usamos regresión. Si fuera una categoría (sí/no, tipo A/B/C), usaríamos clasificación.  
> La regresión lineal asume que la relación entre las features y la variable objetivo es aproximadamente lineal: `y ≈ β₀ + β₁x₁ + β₂x₂ + ...`

## 0. Descarga del dataset

El dataset original está en Kaggle:  
📦 [Student Performance - Multiple Linear Regression](https://www.kaggle.com/datasets/nikhil7280/student-performance-multiple-linear-regression)

**Opciones para obtenerlo:**

**Opción A – Kaggle API (recomendado en Colab):**
```bash
# 1. Sube tu archivo kaggle.json con las credenciales de tu cuenta Kaggle
# 2. Ejecuta:
!pip install kaggle -q
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d nikhil7280/student-performance-multiple-linear-regression --unzip
```

**Opción B – Descarga manual:**  
Descarga `Student_Performance.csv` desde la página de Kaggle y súbelo a la misma carpeta del notebook.

**Opción C – Generar datos sintéticos (para practicar sin Kaggle):**  
Si no tienes acceso a Kaggle, la siguiente celda genera un dataset sintético con la misma estructura.

In [ ]:
# Opción C: datos sintéticos con la misma estructura del dataset de Kaggle
# Comenta/elimina esta celda si tienes el archivo real

import numpy as np
import pandas as pd

np.random.seed(42)
n = 10000

hours_studied = np.random.uniform(1, 9, n)
previous_scores = np.random.uniform(40, 99, n)
extracurricular = np.random.choice(['Yes', 'No'], n, p=[0.4, 0.6])
sleep_hours = np.random.uniform(4, 9, n)
sample_papers = np.random.randint(1, 10, n)

# Performance Index: combinación lineal + ruido
performance_index = (
    2.5 * hours_studied
    + 0.6 * previous_scores
    + 1.5 * (extracurricular == 'Yes').astype(float)
    + 0.4 * sleep_hours
    + 0.8 * sample_papers
    - 15
    + np.random.normal(0, 4, n)
).clip(10, 100)

synthetic_df = pd.DataFrame({
    'Hours Studied': hours_studied.round(1),
    'Previous Scores': previous_scores.round(1),
    'Extracurricular Activities': extracurricular,
    'Sleep Hours': sleep_hours.round(1),
    'Sample Question Papers Practiced': sample_papers,
    'Performance Index': performance_index.round(2)
})

synthetic_df.to_csv('Student_Performance.csv', index=False)
print('✅ Dataset sintético creado: Student_Performance.csv')
print(f'   Filas: {len(synthetic_df):,} | Columnas: {synthetic_df.shape[1]}')

## 1. Importar librerías

Organizamos los imports por categoría para mantener el código limpio y entender qué aporta cada librería.

| Librería | ¿Para qué la usamos? |
|---|---|
| `numpy` / `pandas` | Manipulación de datos numéricos y tablas |
| `train_test_split` | Dividir datos en entrenamiento y prueba |
| `ColumnTransformer` | Aplicar transformaciones distintas por tipo de columna |
| `Pipeline` | Encadenar preprocesamiento + modelo en un solo objeto |
| `StandardScaler` | Escalar variables numéricas (media 0, std 1) |
| `SimpleImputer` | Imputar valores nulos |
| `OneHotEncoder` | Convertir variables categóricas a números |
| `LinearRegression` | El modelo que vamos a entrenar |
| Métricas | Evaluar qué tan bien predice el modelo |

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('✅ Librerías importadas correctamente')

## 2. Cargar los datos

Cargamos el CSV y hacemos una inspección inicial: cuántas filas/columnas tiene, qué tipos de datos hay, y si existen valores nulos.

In [ ]:
df = pd.read_csv('Student_Performance.csv')

print(f'Dimensiones del dataset: {df.shape[0]:,} filas × {df.shape[1]} columnas\n')
df.head()

In [ ]:
# Tipos de datos y valores nulos
print('=== Tipos de datos y nulos ===')
info = pd.DataFrame({
    'dtype': df.dtypes,
    'nulos': df.isnull().sum(),
    '% nulos': (df.isnull().mean() * 100).round(2)
})
print(info)

print('\n=== Estadísticas descriptivas ===')
df.describe(include='all')

In [ ]:
# Exploración visual de la variable objetivo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histograma
axes[0].hist(df['Performance Index'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución del Performance Index', fontsize=13)
axes[0].set_xlabel('Performance Index')
axes[0].set_ylabel('Frecuencia')

# Correlación de numéricas con la variable objetivo
num_cols = df.select_dtypes(include='number').columns.tolist()
corr = df[num_cols].corr()['Performance Index'].drop('Performance Index').sort_values()
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in corr.values]
axes[1].barh(corr.index, corr.values, color=colors)
axes[1].set_title('Correlación con Performance Index', fontsize=13)
axes[1].set_xlabel('Correlación de Pearson')
axes[1].axvline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.show()

print('\n💡 Interpretación: las variables con mayor correlación positiva')
print('   serán más útiles para predecir el Performance Index.')

## 3. Separación de X e y

Separamos las **features** (X) de la **variable objetivo** (y).

- **X** → todo lo que el modelo usará para predecir (características del estudiante)
- **y** → lo que queremos predecir: `Performance Index`

Luego dividimos en **train** (80%) y **test** (20%).  
> ⚠️ El test set **nunca** debe usarse para tomar decisiones de preprocesamiento o ajuste del modelo. Solo para evaluación final.

In [ ]:
TARGET = 'Performance Index'

X = df.drop(columns=[TARGET])
y = df[TARGET]

print(f'Features (X): {X.shape}  →  columnas: {X.columns.tolist()}')
print(f'Objetivo (y): {y.shape}  →  rango: [{y.min():.1f}, {y.max():.1f}]')

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print(f'\nTamaño train: {X_train.shape[0]:,} filas')
print(f'Tamaño test:  {X_test.shape[0]:,} filas')

## 4. Preprocesamiento con ColumnTransformer

El dataset tiene dos tipos de columnas que requieren tratamientos distintos:

| Tipo | Columnas | Transformaciones |
|---|---|---|
| **Numéricas** | Hours Studied, Previous Scores, Sleep Hours, Sample Question Papers Practiced | Imputar con mediana → Escalar con StandardScaler |
| **Categóricas** | Extracurricular Activities | Imputar con moda → OneHotEncoder |

### ¿Por qué escalar en regresión lineal?
La regresión lineal **no requiere** escalado para funcionar matemáticamente, pero escalar:
- Hace que los **coeficientes sean comparables** entre sí (importancia relativa)
- Mejora la **estabilidad numérica** del optimizador
- Es una buena práctica que facilita interpretar el modelo

### ¿Por qué OneHotEncoding?
Los modelos de sklearn solo aceptan números. La columna `Extracurricular Activities` tiene valores `Yes`/`No`, por lo que la convertimos a 0 y 1.

In [ ]:
# Identificar columnas por tipo
numerical_cols = X.select_dtypes(include='number').columns.tolist()
categorical_cols = X.select_dtypes(include='object').columns.tolist()

print(f'Columnas numéricas  ({len(numerical_cols)}): {numerical_cols}')
print(f'Columnas categóricas ({len(categorical_cols)}): {categorical_cols}')

In [ ]:
# Pipeline para variables numéricas
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),   # Rellena nulos con la mediana
    ('scaler', StandardScaler())                      # Escala: (x - media) / std
])

# Pipeline para variables categóricas
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),   # Rellena nulos con la moda
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first'))  # Evita multicolinealidad
])

# ColumnTransformer: aplica cada pipeline al subconjunto de columnas correspondiente
preprocess = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numerical_cols),
    ('cat', categorical_pipeline, categorical_cols)
])

print('✅ ColumnTransformer definido')
print('   Pasos para numéricas:  imputer → scaler')
print('   Pasos para categóricas: imputer → OneHotEncoder')

## 5. Definición del modelo en Pipeline

Encadenamos el preprocesador y el modelo en un único objeto `Pipeline`.  
Esto es fundamental porque:
- Garantiza que la **transformación de test se aprende solo del train**
- Simplifica el código: `pipe.fit(X_train, y_train)` hace todo de una vez
- Facilita el deployment: el pipe es el "modelo completo"

```
X_train  →  [ColumnTransformer]  →  X_transformada  →  [LinearRegression]  →  ŷ
```

In [ ]:
pipe = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', LinearRegression())
])

print('Pipeline definido:')
print(pipe)

## 6. Entrenamiento

Un solo `.fit()` entrena todo el pipeline:  
1. Aprende la media y std de las columnas numéricas (del train)
2. Aprende las categorías de las columnas categóricas (del train)
3. Aplica las transformaciones
4. Ajusta los coeficientes de la regresión lineal

> ⚠️ **Jamás hacer `.fit()` con datos de test.** Solo `.predict()` o `.transform()`.

In [ ]:
pipe.fit(X_train, y_train)

print('✅ Modelo entrenado exitosamente')

# Ver los coeficientes aprendidos
model = pipe.named_steps['model']

# Reconstruir nombres de columnas después del ColumnTransformer
cat_feature_names = (
    pipe.named_steps['preprocess']
    .named_transformers_['cat']
    .named_steps['encoder']
    .get_feature_names_out(categorical_cols)
    .tolist()
)
feature_names = numerical_cols + cat_feature_names

coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coeficiente': model.coef_
}).sort_values('Coeficiente', ascending=False)

print(f'\nIntercepto (β₀): {model.intercept_:.4f}')
print('\nCoeficientes del modelo (sobre datos escalados):')
print(coef_df.to_string(index=False))
print('\n💡 Un coeficiente positivo → más de esa variable = mayor Performance Index.')
print('   Como los datos están escalados, los coeficientes son comparables entre sí.')

## 7. Evaluación del modelo

Para regresión usamos métricas distintas a las de clasificación. Las tres más comunes son:

| Métrica | Fórmula | Interpretación |
|---|---|---|
| **MAE** (Mean Absolute Error) | `mean(|y - ŷ|)` | Error promedio en las mismas unidades que y. Fácil de interpretar. |
| **RMSE** (Root Mean Squared Error) | `sqrt(mean((y - ŷ)²))` | Penaliza más los errores grandes. Mismo rango de unidades que y. |
| **R²** (Coeficiente de determinación) | `1 - SS_res/SS_tot` | Porcentaje de varianza explicada. 1.0 = perfecto, 0 = inútil, <0 = peor que la media. |

> Evaluamos siempre en **test set** para estimar el rendimiento en datos no vistos.

In [ ]:
# Predicciones en train y test
y_pred_train = pipe.predict(X_train)
y_pred_test  = pipe.predict(X_test)

# Calcular métricas
def evaluar(y_true, y_pred, nombre):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f'--- {nombre} ---')
    print(f'  MAE:  {mae:.4f}')
    print(f'  RMSE: {rmse:.4f}')
    print(f'  R²:   {r2:.4f} ({r2*100:.1f}% de la varianza explicada)')
    return mae, rmse, r2

mae_tr, rmse_tr, r2_tr = evaluar(y_train, y_pred_train, 'Train')
print()
mae_te, rmse_te, r2_te = evaluar(y_test, y_pred_test, 'Test')

gap_r2 = r2_tr - r2_te
print(f'\n📊 Brecha R² (train - test): {gap_r2:.4f}')
if gap_r2 < 0.02:
    print('   → Muy baja: el modelo generaliza bien (sin sobreajuste notable)')
elif gap_r2 < 0.05:
    print('   → Moderada: hay algo de sobreajuste, considera regularización')
else:
    print('   → Alta: el modelo está sobreajustado al train')

In [ ]:
# Gráficas de evaluación
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Predichos vs Reales
axes[0].scatter(y_test, y_pred_test, alpha=0.3, s=15, color='steelblue')
lim = [y_test.min(), y_test.max()]
axes[0].plot(lim, lim, 'r--', linewidth=1.5, label='Predicción perfecta')
axes[0].set_xlabel('Valores reales (y)')
axes[0].set_ylabel('Valores predichos (ŷ)')
axes[0].set_title(f'Real vs Predicho\nR² = {r2_te:.3f}', fontsize=12)
axes[0].legend()

# 2. Distribución de residuos
residuals = y_test - y_pred_test
axes[1].hist(residuals, bins=40, color='coral', edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_xlabel('Residuo (y - ŷ)')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title(f'Distribución de Residuos\nMedia: {residuals.mean():.2f}, Std: {residuals.std():.2f}', fontsize=12)

# 3. Residuos vs Predichos (para detectar patrones no lineales)
axes[2].scatter(y_pred_test, residuals, alpha=0.3, s=15, color='mediumpurple')
axes[2].axhline(0, color='red', linestyle='--', linewidth=1)
axes[2].set_xlabel('Valores predichos (ŷ)')
axes[2].set_ylabel('Residuo (y - ŷ)')
axes[2].set_title('Residuos vs Predichos\n(no debería verse un patrón)', fontsize=12)

plt.tight_layout()
plt.show()

print('\n💡 Interpretación de las gráficas:')
print('  • Real vs Predicho: los puntos deben estar cerca de la línea roja')
print('  • Residuos: deben ser aproximadamente normales con media ≈ 0')
print('  • Residuos vs Predichos: no debe haber forma de embudo ni curva (indicaría no linealidad)')

In [ ]:
# Visualización de coeficientes (importancia de cada feature)
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in coef_df['Coeficiente']]
ax.barh(coef_df['Feature'], coef_df['Coeficiente'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Magnitud del coeficiente (datos escalados)')
ax.set_title('Importancia de las variables\n(coeficientes de la Regresión Lineal)', fontsize=12)
plt.tight_layout()
plt.show()

print('\n💡 Las barras más largas indican mayor influencia sobre el Performance Index.')
print('   Verde = efecto positivo | Rojo = efecto negativo')

## 8. Ejercicios propuestos (para entregar)

La idea NO es "adivinar", sino **cambiar una cosa, medir, y explicar con evidencia**.

---

### Ejercicio 1 — Quitar el escalado
Elimina `StandardScaler()` del pipeline numérico y compara los coeficientes y las métricas.  
Preguntas: ¿Cambia el R²? ¿Cambian los coeficientes? ¿Por qué son ahora difíciles de comparar?

### Ejercicio 2 — Agregar una feature nueva (Ingeniería de características)
Crea una nueva columna antes de entrenar, por ejemplo:  
`df['Study_x_Papers'] = df['Hours Studied'] * df['Sample Question Papers Practiced']`  
¿Mejora el R²? ¿Tiene sentido este nuevo feature?

### Ejercicio 3 — Regresión con regularización (Ridge y Lasso)
Reemplaza `LinearRegression()` por `Ridge(alpha=1.0)` y por `Lasso(alpha=0.1)`.  
Compara MAE, RMSE y R² de los tres modelos. ¿Cuándo conviene usar regularización?

### Ejercicio 4 — Cambiar la estrategia de imputación
Cambia `strategy='median'` a `strategy='mean'` en el imputer numérico.  
¿Cambian las métricas? ¿Cuándo es más robusta la mediana vs la media?

### Ejercicio 5 — Analizar el modelo cualitativamente
Basándote en los coeficientes del modelo:  
a) ¿Qué variable impacta más el rendimiento?  
b) ¿Tiene sentido que `Extracurricular Activities` sea positiva o negativa?  
c) ¿Hay algún coeficiente que te sorprenda? Justifica.

In [ ]:
# ============================================================
# Plantilla Ejercicio 3 — Ridge y Lasso
# ============================================================
from sklearn.linear_model import Ridge, Lasso

modelos = {
    'LinearRegression': LinearRegression(),
    'Ridge (alpha=1)': Ridge(alpha=1.0),
    'Lasso (alpha=0.1)': Lasso(alpha=0.1),
}

print(f'{"Modelo":<25} {"MAE":>8} {"RMSE":>8} {"R²":>8}')
print('-' * 55)

for nombre, modelo in modelos.items():
    pipe_temp = Pipeline(steps=[
        ('preprocess', preprocess),
        ('model', modelo)
    ])
    pipe_temp.fit(X_train, y_train)
    y_pred_temp = pipe_temp.predict(X_test)

    mae  = mean_absolute_error(y_test, y_pred_temp)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_temp))
    r2   = r2_score(y_test, y_pred_temp)
    print(f'{nombre:<25} {mae:>8.4f} {rmse:>8.4f} {r2:>8.4f}')

print('\n💡 ¿Cuál modelo funciona mejor? ¿Por qué?')
print('   Escribe tu análisis aquí...')

In [ ]:
# ============================================================
# Plantilla Ejercicio 1 — Sin escalado
# ============================================================

numeric_pipeline_sin_scaler = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
    # Sin StandardScaler
])

preprocess_sin_scaler = ColumnTransformer(transformers=[
    ('num', numeric_pipeline_sin_scaler, numerical_cols),
    ('cat', categorical_pipeline, categorical_cols)
])

pipe_sin_scaler = Pipeline(steps=[
    ('preprocess', preprocess_sin_scaler),
    ('model', LinearRegression())
])

pipe_sin_scaler.fit(X_train, y_train)
y_pred_sin = pipe_sin_scaler.predict(X_test)

print('=== CON StandardScaler ===')
print(f'  R²:  {r2_score(y_test, y_pred_test):.4f}')
print(f'  MAE: {mean_absolute_error(y_test, y_pred_test):.4f}')

print('\n=== SIN StandardScaler ===')
print(f'  R²:  {r2_score(y_test, y_pred_sin):.4f}')
print(f'  MAE: {mean_absolute_error(y_test, y_pred_sin):.4f}')

# Coeficientes sin scaler
coef_sin = pipe_sin_scaler.named_steps['model'].coef_
print(f'\nCoeficientes sin scaler: {dict(zip(numerical_cols, coef_sin[:len(numerical_cols)]))}')
print('\n💡 ¿Cambia el R²? ¿Por qué los coeficientes son ahora más difíciles de comparar?')
print('   Escribe tu análisis aquí...')